In [2]:
import os
import pandas as pd
import numpy as np 
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

C:\Users\marie\AppData\Local\Temp\ipykernel_26420\1757850363.py:2: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd


In [3]:
dirpath = os.getcwd()
features_path = r"data\gmfeature_table.csv"
data_path = r"C:\Users\marie\rep_codes\udder_project\udder_analysis\long_format_df"
visit_path = r"C:\Users\marie\rep_codes\udder_project\delpro_vms\data\milk_videos_visit.csv"
plot_dir = os.path.join(os.path.normpath(dirpath + os.sep + os.pardir),r"adsa\examples")

In [5]:
df = pd.read_csv(os.path.join(data_path, "lactation_features.csv"))
vdf = pd.read_csv(visit_path)
vdf_selected = vdf[['cow', 'days_in_milk']]
df_merged = df.merge(vdf_selected, on = 'cow')
len(np.unique(df_merged.cow))

138

In [6]:
# add min teat length, max teat length, min eu distance, max eu distance, min gd distance, max gd distance
df_merged["min_teat"] = [np.nanmin(df_merged.loc[i, ["len_rf", "len_rb", "len_lf", "len_lb"]].values.astype('float')) for i in df_merged.index]
df_merged["max_teat"]= [np.nanmax(df_merged.loc[i, ["len_rf", "len_rb", "len_lf", "len_lb"]].values.astype('float')) for i in df_merged.index]
df_merged["min_eu"] = [np.nanmin(df_merged.loc[i, ["eu_front", "eu_right", "eu_back", "eu_left"]].values.astype('float')) for i in df_merged.index]
df_merged["max_eu"] = [np.nanmax(df_merged.loc[i, ["eu_front", "eu_right", "eu_back", "eu_left"]].values.astype('float')) for i in df_merged.index]
df_merged["min_gd"] = [np.nanmin(df_merged.loc[i, ["gd_front", "gd_right", "gd_back", "gd_left"]].values.astype('float')) for i in df_merged.index]
df_merged["max_gd"] = [np.nanmax(df_merged.loc[i, ["gd_front", "gd_right", "gd_back", "gd_left"]].values.astype('float')) for i in df_merged.index]

C:\Users\marie\AppData\Local\Temp\ipykernel_26420\1974453648.py:2: RuntimeWarning: All-NaN slice encountered
  df_merged["min_teat"] = [np.nanmin(df_merged.loc[i, ["len_rf", "len_rb", "len_lf", "len_lb"]].values.astype('float')) for i in df_merged.index]
C:\Users\marie\AppData\Local\Temp\ipykernel_26420\1974453648.py:3: RuntimeWarning: All-NaN slice encountered
  df_merged["max_teat"]= [np.nanmax(df_merged.loc[i, ["len_rf", "len_rb", "len_lf", "len_lb"]].values.astype('float')) for i in df_merged.index]
C:\Users\marie\AppData\Local\Temp\ipykernel_26420\1974453648.py:6: RuntimeWarning: All-NaN slice encountered
  df_merged["min_gd"] = [np.nanmin(df_merged.loc[i, ["gd_front", "gd_right", "gd_back", "gd_left"]].values.astype('float')) for i in df_merged.index]
C:\Users\marie\AppData\Local\Temp\ipykernel_26420\1974453648.py:6: RuntimeWarning: All-NaN slice encountered
  df_merged["min_gd"] = [np.nanmin(df_merged.loc[i, ["gd_front", "gd_right", "gd_back", "gd_left"]].values.astype('float'))

In [7]:
udder_features = ['vol_udder', 'sarea_udder', 'peri_udder', 'area_udder', 'circ_udder', 'exc_udder','min_teat', 'max_teat', 'min_eu', 'max_eu', 'min_gd', 'max_gd']
prod_vars = ['yield_visit_mean', 'interval_sec_mean', 'kickoff_any_perc', 'days_in_milk', "lactation"]

In [10]:
udder_pearson_df = pd.DataFrame(index = udder_features, columns = prod_vars)
udder_pvals_df = pd.DataFrame(index = udder_features, columns = prod_vars)
udder_ci_df = pd.DataFrame(index = udder_features, columns = prod_vars)
rng = np.random.default_rng()
method = stats.BootstrapMethod(method='BCa', random_state=rng)

In [11]:
for u in udder_features:
    for v in prod_vars:
        selected = df_merged[[v, u]].dropna(axis=0) 
        res = stats.pearsonr(selected[u], selected[v])
        udder_pearson_df.loc[u, v] = res.statistic
        udder_pvals_df.loc[u, v] = res.pvalue
        udder_ci_df.loc[u, v] = res.confidence_interval(confidence_level=0.95, method=method)

In [12]:
udder_pearson_df

,yield_visit_mean,interval_sec_mean,kickoff_any_perc,days_in_milk,lactation
vol_udder,0.450192,0.113585,-0.078037,-0.009584,0.294832
sarea_udder,0.463962,-0.047121,-0.135635,-0.056288,0.383631
peri_udder,0.561599,-0.071791,-0.228523,0.016823,0.724239
area_udder,0.545836,-0.073253,-0.205208,-0.019349,0.700214
circ_udder,-0.360714,0.02432,0.058739,-0.077752,-0.357731
exc_udder,0.030659,0.065105,0.025876,-0.192392,0.078599
min_teat,0.201591,-0.033316,-0.026244,0.045487,0.3126
max_teat,0.132533,-0.02568,-0.010383,-0.019914,0.161203
min_eu,0.139468,0.098477,0.149778,-0.131401,0.120178
max_eu,0.377546,-0.046422,-0.168759,0.052241,0.47995


In [14]:
udder_pvals_df

,yield_visit_mean,interval_sec_mean,kickoff_any_perc,days_in_milk,lactation
vol_udder,0.0,0.194706,0.373784,0.913148,0.0006
sarea_udder,0.0,0.588742,0.118144,0.51829,0.000005
peri_udder,0.0,0.406221,0.007452,0.84587,0.0
area_udder,0.0,0.396706,0.016549,0.82308,0.0
circ_udder,0.000039,0.788614,0.516968,0.390705,0.000045
exc_udder,0.73119,0.46531,0.77187,0.029579,0.377835
min_teat,0.016525,0.694917,0.757388,0.59224,0.000161
max_teat,0.117196,0.762447,0.902745,0.814688,0.056179
min_eu,0.09785,0.243634,0.075227,0.119055,0.154274
max_eu,0.000004,0.583285,0.044682,0.536942,0.0


In [15]:
udder_pvals_adj_df = udder_pvals_df.copy()
col_num = udder_pvals_adj_df.shape[1]
for col in range(col_num):
    ps = udder_pvals_adj_df.iloc[:, col].to_list()
    udder_pvals_adj_df.iloc[:, col] = stats.false_discovery_control(ps, method='bh') 

In [16]:
udder_pvals_adj_df

,yield_visit_mean,interval_sec_mean,kickoff_any_perc,days_in_milk,lactation
vol_udder,0.0,0.788614,0.587754,0.913148,0.0009
sarea_udder,0.0,0.788614,0.283547,0.88836,0.000011
peri_udder,0.0,0.788614,0.089428,0.913148,0.0
area_udder,0.0,0.788614,0.099291,0.913148,0.0
circ_udder,0.000066,0.788614,0.689291,0.88836,0.00009
exc_udder,0.73119,0.788614,0.84204,0.354949,0.377835
min_teat,0.024787,0.788614,0.84204,0.88836,0.000276
max_teat,0.12785,0.788614,0.902745,0.913148,0.067415
min_eu,0.11742,0.788614,0.22568,0.714328,0.168298
max_eu,0.000007,0.788614,0.178727,0.88836,0.0


In [17]:
udder_ci_df

,yield_visit_mean,interval_sec_mean,kickoff_any_perc,days_in_milk,lactation
vol_udder,"(0.30680840512667484, 0.5627846819520811)","(-0.05456742554914582, 0.269380314944653)","(-0.23052974624981537, 0.04637492206568684)","(-0.17071404668902165, 0.1539107860624076)","(0.12865497064251574, 0.45047767074141293)"
sarea_udder,"(0.33801988269277594, 0.5761469387587015)","(-0.19301891592623077, 0.10616228296309183)","(-0.26561737680014846, 0.04738058927671565)","(-0.2082142435101232, 0.09839962822678276)","(0.20601342797511454, 0.5016064727041939)"
peri_udder,"(0.43007312564298694, 0.6601542313522623)","(-0.22731056699585064, 0.0846823251062138)","(-0.39112967572359053, -0.04073807485978889)","(-0.15161752545557214, 0.19899495991572042)","(0.6325712962260581, 0.7861967681308268)"
area_udder,"(0.41536526738461627, 0.648011765548315)","(-0.23419804577342176, 0.09700991155473467)","(-0.3502620125364698, -0.013385602654235438)","(-0.18711693034534055, 0.1689920900953201)","(0.6006341820218755, 0.7717179555615951)"
circ_udder,"(-0.4909155942338463, -0.20264921150056023)","(-0.13242428206355666, 0.18760798523853414)","(-0.049756994178507104, 0.17475922430402746)","(-0.23558011314560193, 0.09351751486210048)","(-0.4974332255855038, -0.18299255995896668)"
exc_udder,"(-0.1433025137696702, 0.20008238703316014)","(-0.13654671215032801, 0.2224458686888602)","(-0.22080175335179836, 0.17931414384589064)","(-0.345824372220269, -0.029085903975378854)","(-0.078826990986834, 0.2447461766071304)"
min_teat,"(0.052358366132902956, 0.34620163206719484)","(-0.1746678459886239, 0.11074055293047486)","(-0.19909699484013946, 0.10312962181740852)","(-0.11403171151932345, 0.19403473730032322)","(0.17223882503167473, 0.4465863580804383)"
max_teat,"(-0.024669860426357607, 0.29225210896638804)","(-0.15703817375663182, 0.11256034686642634)","(-0.1217885252670729, 0.07346437649863065)","(-0.1689642928690023, 0.13517946428422406)","(0.012129071639821333, 0.3027221726354981)"
min_eu,"(-0.015238742946459938, 0.2824266729493318)","(-0.08014737239537265, 0.23975868368043324)","(0.012343262398317349, 0.3247009502937978)","(-0.28095986871370654, 0.018756062738392244)","(-0.03862444828899135, 0.28115558688432507)"
max_eu,"(0.16594071639953004, 0.5137214610386793)","(-0.22171757981882007, 0.11794049410334649)","(-0.25087080244653054, -0.07505273984696538)","(-0.11114453764572184, 0.2127163766423569)","(0.26548571788653186, 0.5994639397645976)"


In [18]:
udder_pvals_df.to_csv(os.path.join(dirpath, "tables", "udder_pvals_df.csv"), index = True)
udder_pearson_df.to_csv(os.path.join(dirpath, "tables", "udder_pearson_df.csv"), index = True)
udder_pvals_adj_df.to_csv(os.path.join(dirpath, "tables", "udder_pvals_adj_df.csv"), index = True)
udder_ci_df.to_csv(os.path.join(dirpath, "tables", "udder_ci_df.csv"), index = True)